## TASK 2

In [1]:
%pip install autogluon

In [5]:
import pandas as pd
import autogluon.common as ag
from autogluon.tabular import TabularDataset, TabularPredictor
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Load the Iris dataset and construct a DataFrame
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target

# Split the data into train, validation, and test subsets
train_val, val_set = train_test_split(df, test_size=0.1, stratify=df['target'], random_state=42)
train_set, test_set = train_test_split(train_val, test_size=0.2 / 0.9, stratify=train_val['target'], random_state=42)

# Convert DataFrames to AutoGluon TabularDataset format
train_data = TabularDataset(train_set)
val_data   = TabularDataset(val_set)
test_data  = TabularDataset(test_set)
true_labels = test_data['target']

def build_predictor(search_method: str) -> TabularPredictor:
    """
    Constructs and fits a TabularPredictor using hyperparameter tuning.

    Parameters:
        search_method (str): The search strategy for hyperparameter tuning ('random' or 'bayes').

    Returns:
        A fitted TabularPredictor.
    """
    predictor = TabularPredictor(label='target', problem_type='multiclass', eval_metric='accuracy')
    return predictor.fit(
        train_data=train_data,
        tuning_data=val_data,
        hyperparameters={
            'NN_TORCH': {
                'batch_size': ag.space.Categorical(2, 4),
                'num_epochs': ag.space.Categorical(1, 3, 5),
                'learning_rate': ag.space.Categorical(1e-3, 1e-5),
                'num_layers': 1,
                'hidden_size': 16,
                'activation': 'relu'
            }
        },
        hyperparameter_tune_kwargs={
            'num_trials': 10,
            'scheduler': 'local',
            'searcher': search_method
        },
        num_bag_folds=5,
        num_stack_levels=0,
        use_bag_holdout=True,
        time_limit=100
    )

def evaluate_model(predictor: TabularPredictor, dataset: TabularDataset, description: str):
    """
    Evaluates the predictor on the provided dataset and prints metrics and leaderboard.

    Parameters:
        predictor (TabularPredictor): The model to evaluate.
        dataset (TabularDataset): The dataset for evaluation.
        description (str): A short description to identify the search strategy.
    """
    predictions = predictor.predict(dataset.drop(columns=['target']))
    acc = accuracy_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions, average='weighted')

    print(description)
    print(f"Test Accuracy: {acc:.4f}")
    print(f"Test F1: {f1:.4f}")

    lb = predictor.leaderboard(dataset)
    print("Leaderboard:")
    print(lb[['model', 'score_test', 'fit_time']])




In [6]:
# Create and evaluate a predictor using random search
random_pred = build_predictor('random')
evaluate_model(random_pred, test_data, "Random Search:")



No path specified. Models will be saved in: "AutogluonModels/ag-20250226_183702"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.11.11
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun 27 21:05:47 UTC 2024
CPU Count:          2
Memory Avail:       10.41 GB / 12.67 GB (82.1%)
Disk Space Avail:   71.16 GB / 107.72 GB (66.1%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='experimental' : New in v1.2: Pre-trained foundation model + parallel fits. The absolute best accuracy without consideration for inference speed. Does not support GPU.
	presets='best'         : Maximize accuracy. Recommended for most users. Use in competitions 

+----------------------------------------------------------+
| Configuration for experiment     NeuralNetTorch_BAG_L1   |
+----------------------------------------------------------+
| Search algorithm                 BasicVariantGenerator   |
| Scheduler                        FIFOScheduler           |
| Number of trials                 10                      |
+----------------------------------------------------------+

View detailed results here: /content/AutogluonModels/ag-20250226_183702/models/NeuralNetTorch_BAG_L1


Fitted model: NeuralNetTorch_BAG_L1/abfcc_00000 ...
	0.4904	 = Validation score   (accuracy)
	19.56s	 = Training   runtime
	0.02s	 = Validation runtime
Fitted model: NeuralNetTorch_BAG_L1/abfcc_00001 ...
	0.4615	 = Validation score   (accuracy)
	20.16s	 = Training   runtime
	0.04s	 = Validation runtime
Fitted model: NeuralNetTorch_BAG_L1/abfcc_00002 ...
	0.6731	 = Validation score   (accuracy)
	19.7s	 = Training   runtime
	0.02s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 99.95s of the 4.22s of remaining time.
	Ensemble Weights: {'NeuralNetTorch_BAG_L1/abfcc_00002': 1.0}
	0.6667	 = Validation score   (accuracy)
	0.0s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training complete, total runtime = 95.81s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 817.1 rows/s (15 batch size)
TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/content/AutogluonModels/ag-20250226_183702")



Random Search:
Test Accuracy: 0.6452
Test F1: 0.5451
Leaderboard:
                               model  score_test   fit_time
0  NeuralNetTorch_BAG_L1/abfcc_00002    0.645161  19.704742
1                WeightedEnsemble_L2    0.645161  19.709050
2  NeuralNetTorch_BAG_L1/abfcc_00001    0.483871  20.156523
3  NeuralNetTorch_BAG_L1/abfcc_00000    0.354839  19.562698


In [7]:
# Create and evaluate a predictor using Hyperband Search with Bayesian optimization
bayes_pred = build_predictor('bayes')
evaluate_model(bayes_pred, test_data, "Hyperband Search + Bayes Optimization:")

No path specified. Models will be saved in: "AutogluonModels/ag-20250226_183838"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.11.11
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun 27 21:05:47 UTC 2024
CPU Count:          2
Memory Avail:       9.95 GB / 12.67 GB (78.5%)
Disk Space Avail:   71.15 GB / 107.72 GB (66.1%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='experimental' : New in v1.2: Pre-trained foundation model + parallel fits. The absolute best accuracy without consideration for inference speed. Does not support GPU.
	presets='best'         : Maximize accuracy. Recommended for most users. Use in competitions a

+----------------------------------------------------------+
| Configuration for experiment     NeuralNetTorch_BAG_L1   |
+----------------------------------------------------------+
| Search algorithm                 SearchGenerator         |
| Scheduler                        FIFOScheduler           |
| Number of trials                 10                      |
+----------------------------------------------------------+

View detailed results here: /content/AutogluonModels/ag-20250226_183838/models/NeuralNetTorch_BAG_L1


Fitted model: NeuralNetTorch_BAG_L1/87ff57d9 ...
	0.4904	 = Validation score   (accuracy)
	19.52s	 = Training   runtime
	0.03s	 = Validation runtime
Fitted model: NeuralNetTorch_BAG_L1/dc1029ef ...
	0.1442	 = Validation score   (accuracy)
	20.42s	 = Training   runtime
	0.04s	 = Validation runtime
Fitted model: NeuralNetTorch_BAG_L1/6b134068 ...
	0.7212	 = Validation score   (accuracy)
	21.05s	 = Training   runtime
	0.04s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 99.95s of the 7.65s of remaining time.
	Ensemble Weights: {'NeuralNetTorch_BAG_L1/6b134068': 1.0}
	0.6667	 = Validation score   (accuracy)
	0.0s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training complete, total runtime = 92.37s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 514.8 rows/s (15 batch size)
TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/content/AutogluonModels/ag-20250226_183838")



Hyperband Search + Bayes Optimization:
Test Accuracy: 0.6774
Test F1: 0.6629
Leaderboard:
                            model  score_test   fit_time
0  NeuralNetTorch_BAG_L1/6b134068    0.677419  21.047420
1             WeightedEnsemble_L2    0.677419  21.051544
2  NeuralNetTorch_BAG_L1/87ff57d9    0.354839  19.516655
3  NeuralNetTorch_BAG_L1/dc1029ef    0.032258  20.417789


# Impact of Hyperparameters on Model Performance

**Epochs**:
Increasing the number of epochs generally improves model performance because it allows the model more iterations to learn from the data. This trend is visible in the loss-versus-epochs plots, where a higher epoch count corresponds with lower loss values, indicating better learning.

**Batch Size**:
A smaller batch size tends to enhance performance, as it leads to more frequent weight updates during training. In contrast, larger batch sizes may result in higher loss values, suggesting that the model might be less responsive to subtle patterns in the data.

**Learning Rate:**
A higher learning rate can help the model converge faster, often resulting in lower loss values. This relationship is observed across different hyperparameter settings, where increasing the learning rate generally correlates with improved performance.

# Manual Tuning vs. Automated Search
**Automated Search:**
Automated hyperparameter tuning is particularly advantageous for large, complex models. It efficiently explores a broad range of parameter combinations, often achieving superior results while reducing the time and effort required compared to manual tuning.

**Manual Tuning**:
Manual tuning can be beneficial when you possess strong domain knowledge and intuition about the problem. It may be a more practical option for smaller models or when computational resources are limited, even though it might not explore the parameter space as thoroughly as automated methods.